In [1]:
# Install TensorFlow and Streamlit
!pip install tensorflow streamlit


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.3/44.3 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 52.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 51.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.1/79.1 kB 4.6 MB/s eta 0:00:00


In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.preprocessing import sequence
import numpy as np

# Load IMDb dataset
(train_data, train_labels), (test_data, test_labels) = tf.keras.datasets.imdb.load_data(num_words=10000)

# Pad sequences to make them of equal length
max_len = 500  # Maximum length of the sequences
train_data = sequence.pad_sequences(train_data, maxlen=max_len)
test_data = sequence.pad_sequences(test_data, maxlen=max_len)

# Prepare labels as numpy arrays
train_labels = np.array(train_labels)
test_labels = np.array(test_labels)

# Define the model
model = models.Sequential()


embedding_dim = 128
model.add(layers.Embedding(input_dim=10000, output_dim=embedding_dim, input_length=max_len))

# LSTM layer (instead of SimpleRNN) with dropout for regularization
model.add(layers.LSTM(256, activation='tanh', dropout=0.2, recurrent_dropout=0.2))

# Dense output layer with sigmoid activation for binary classification
model.add(layers.Dense(1, activation='sigmoid'))

# Compile the model
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])


model.build(input_shape=(None, max_len))
model.summary()


# EarlyStopping callback to stop training if validation loss does not improve
early_stopping = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

# Train the model with early stopping
history = model.fit(
    train_data, train_labels,
    epochs=20,
    batch_size=64,
    validation_data=(test_data, test_labels),
    callbacks=[early_stopping]

# Evaluate the model on the test data
test_loss, test_accuracy = model.evaluate(test_data, test_labels, batch_size=64)

# Print the test accuracy
print(f"Test Accuracy: {test_accuracy * 100:.2f}%")


In [5]:
# Save the model
model.save('/content/imdb_rnn_model.h5')


In [6]:
# Create the app.py file
with open("app.py", "w") as f:
    f.write("""
import streamlit as st
import tensorflow as tf
import numpy as np
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Load the pre-trained model
model = tf.keras.models.load_model('imdb_rnn_model.h5')

# Function to predict sentiment
def predict_sentiment(text):
    # Tokenize and pad the text
    tokenizer = tf.keras.preprocessing.text.Tokenizer(num_words=10000)
    tokenizer.fit_on_texts([text])
    sequence = tokenizer.texts_to_sequences([text])
    padded_sequence = pad_sequences(sequence, maxlen=256, padding='post')

    # Predict the sentiment
    prediction = model.predict(padded_sequence)
    if prediction >= 0.5:
        return 'Positive'
    else:
        return 'Negative'

# Streamlit user interface
st.title('IMDb Movie Review Sentiment Analysis')

review_text = st.text_area("Enter movie review:")

if st.button('Predict Sentiment'):
    if review_text:
        sentiment = predict_sentiment(review_text)
        st.write(f"Sentiment: {sentiment}")
    else:
        st.write("Please enter a review to analyze.")

""")


In [7]:
!pip install ngrok pyngrok

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 39.1 MB/s eta 0:00:00


In [8]:
from pyngrok import ngrok
import os

# Set the port for Streamlit to run
port = 8501

ngrok.set_auth_token('')

# Open a ngrok tunnel to the streamlit port
public_url = ngrok.connect(port)

# Run Streamlit in the background
os.system("streamlit run app.py &")

# Output the public URL
print(f"Streamlit app is running at: {public_url}")


Streamlit app is running at: NgrokTunnel: "https://5bd2-34-168-81-75.ngrok-free.app" -> "http://localhost:8501"
